# 05_Text2Trade_Indexacion_NANDINA

## Propósito
Este notebook construye y persiste un índice de **recuperación semántica densa** (dense retrieval) inspirado en el enfoque **Text2Trade** para recuperar candidatos **NANDINA (8 dígitos)** a partir de una descripción textual (p. ej., descripción de mercancía en una DAM).

En el diseño experimental del proyecto, este índice constituye el **Baseline semántico** (complementario al Baseline léxico BM25) y su salida se utiliza para recuperar un conjunto Top-*k* de subpartidas candidatas que luego pueden ser justificadas mediante técnicas RAG/LLM en etapas posteriores.

## Alcance
- Indexa exclusivamente documentos con `tipo == "nandina_8"`.
- Valida que `codigo` tenga exactamente **8 dígitos**.
- Construye embeddings con un bi-encoder (SentenceTransformer).
- Crea un índice ANN (por defecto **HNSW** vía `hnswlib`, compatible con Windows) y guarda artefactos para reutilización.
- Incluye una prueba rápida (sanity check) de consulta determinista y otra con **Monte Carlo Dropout (MCD)** para estimar estabilidad/incertidumbre (alineado al enfoque Text2Trade).

## Reproducibilidad y auditabilidad
El notebook guarda:
- el modelo (o referencia del modelo base),
- el índice vectorial,
- el *docstore* (texto + metadatos por NANDINA),
- configuración de recuperación (Top-*k*, parámetros MCD),
- un `run_metadata.json` con hashes SHA-256, versión de librerías y parámetros.


In [ ]:
######################################################################################################################
# Notebook: 05_Text2Trade_Indexacion_NANDINA
# Autor del código: Vladimir Molleapasa Gutierrez
# Fecha de generación: 17/01/2026
# Código generado con asistencia de ChatGPT (modelo GPT-5.2 Thinking)
# Uso: académico, con validación y revisión del autor.
#
# OBJETIVO
# Construir y persistir un índice de recuperación semántica (dense retrieval) para subpartidas NANDINA (8 dígitos),
# inspirado en Text2Trade (bi-encoder + cuantificación de incertidumbre mediante Monte Carlo Dropout).
#
# ENTRADAS
# - experiment_config.json (en src/configs) para rutas base.
# - Corpus JSONL procesado: data/processed/corpus_rag_v1_index.jsonl (por defecto) con campos esperados:
#     * tipo, codigo, titulo, texto (o variantes, ver TEXT_FIELD_CANDIDATES)
#
# SALIDAS (ARTEFACTOS)
# - Directorio: data/processed/indexes/text2trade_nandina8_v1/
#     model/    : modelo SentenceTransformer (serializado)
#     index/    : índice ANN (HNSW) y mapping de ids
#     store/    : docstore JSONL (documentos NANDINA-8 indexados)
#     eval/     : salidas de prueba (sanity check)
#     retrieval_config.json
#     text2trade_nandina8_run_metadata.json
#
# REPRODUCIBILIDAD
# - Hash SHA-256 del corpus de entrada y del archivo de configuración.
# - Snapshot del entorno (Python, OS, versiones de librerías).
# - Parámetros fijos de indexación documentados ex ante.
######################################################################################################################


In [5]:
# =========================
# IMPORTS
# =========================

import os
import re
import json
import time
import math
import hashlib
import platform
from pathlib import Path
from typing import Dict, List, Any, Tuple, Optional

# Dependencias principales (dense retrieval)
# - sentence-transformers
# - torch
# - hnswlib (recomendado en Windows)
# - numpy

import numpy as np

# Opcional: pandas para visualización de resultados
try:
    import pandas as pd
except Exception:
    pd = None

# Torch / SentenceTransformers
try:
    import torch
    from sentence_transformers import SentenceTransformer
except Exception as e:
    raise RuntimeError(
        "Faltan dependencias. Instale: sentence-transformers y torch. "
        "Ejemplo: pip install -U sentence-transformers torch"
    ) from e

# Índice ANN: preferir hnswlib (compatible con Windows). Si no está disponible, se usará búsqueda exacta.
try:
    import hnswlib
    HNSW_AVAILABLE = True
except Exception:
    HNSW_AVAILABLE = False


In [6]:
# =========================
# CONFIGURACIÓN (reutiliza experiment_config.json)
# =========================

# 1) Ubicar experiment_config.json.
#    Por defecto, se asume el layout del proyecto:
#      <BASE_DIR>/src/configs/experiment_config.json
#    Si lo ejecuta desde otra ubicación, ajuste CONFIG_PATH.

DEFAULT_BASE_DIR = Path(r"C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código")
CONFIG_PATH_CANDIDATES = [
    DEFAULT_BASE_DIR / "src" / "configs" / "experiment_config.json",
    Path.cwd() / "src" / "configs" / "experiment_config.json",
    Path.cwd().parent / "src" / "configs" / "experiment_config.json",
]

CONFIG_PATH = None
for p in CONFIG_PATH_CANDIDATES:
    if p.exists():
        CONFIG_PATH = p
        break

if CONFIG_PATH is None:
    raise FileNotFoundError(
        "No se encontró experiment_config.json. "
        "Ajuste DEFAULT_BASE_DIR o establezca CONFIG_PATH manualmente. "
        f"Candidatos probados: {CONFIG_PATH_CANDIDATES}"
    )

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    EXP_CONFIG = json.load(f)

BASE_DIR = Path(EXP_CONFIG["paths"]["base_dir"])
DATA_PROCESSED_DIR = BASE_DIR / "data" / "processed"
INDEXES_DIR = DATA_PROCESSED_DIR / "indexes"

# Corpus de trabajo (por defecto, el corpus curado para indexación)
# Nota: si su archivo se llama distinto, ajuste CORPUS_FILE.
CORPUS_FILE = "corpus_rag_v1_index.jsonl"
CORPUS_PATH = DATA_PROCESSED_DIR / CORPUS_FILE

# Parámetros del retriever semántico (baseline)
TARGET_TYPE = "nandina_8"
TYPE_FIELD = "tipo"
CODE_FIELD = "codigo"
TITLE_FIELD = "titulo"

# Campos candidatos para texto (robustez ante cambios de esquema)
TEXT_FIELD_CANDIDATES = [
    "texto_index",  # si existe un campo curado para indexación
    "texto",
    "text",
    "content",
    "descripcion",
]

# Nombre del artefacto (versionado)
ARTIFACT_NAME = "text2trade_nandina8_v1"
OUT_DIR = INDEXES_DIR / ARTIFACT_NAME

# Subdirectorios (se crean si no existen)
MODEL_DIR = OUT_DIR / "model"
INDEX_DIR = OUT_DIR / "index"
STORE_DIR = OUT_DIR / "store"
EVAL_DIR = OUT_DIR / "eval"

# Archivo de configuración de recuperación (queda congelado junto al artefacto)
RETRIEVAL_CONFIG_PATH = OUT_DIR / "retrieval_config.json"
RUN_METADATA_PATH = OUT_DIR / "text2trade_nandina8_run_metadata.json"

# Modelo base recomendado para español/multilingüe (ajustable)
# Nota: En Text2Trade el bi-encoder se fine-tunea. Aquí se inicia con un modelo preentrenado como baseline reproducible.
BASE_BIENCODER = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Parámetros de embedding
BATCH_SIZE = 32
EMBEDDING_NORMALIZE = True  # necesario para coseno vía dot product

# Parámetros de índice ANN (HNSW)
HNSW_SPACE = "cosine"
HNSW_M = 64
HNSW_EF_CONSTRUCTION = 200
HNSW_EF_SEARCH = 200

# Parámetros MCD (Monte Carlo Dropout) para estabilidad/incertidumbre
MCD_ENABLED = True
MCD_PASSES = 50
MCD_TOPK = 50
MCD_TOP3_FREQ_WEIGHT = 0.2
MCD_MEAN_SIM_WEIGHT = 0.8
MCD_BASE_SEED = 123

# Validaciones básicas
if not CORPUS_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el corpus en: {CORPUS_PATH}. "
        "Verifique el nombre del archivo CORPUS_FILE o la carpeta data/processed."
    )

# Crear directorios (idempotente)
for d in [OUT_DIR, MODEL_DIR, INDEX_DIR, STORE_DIR, EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("CONFIG_PATH:", CONFIG_PATH)
print("BASE_DIR:", BASE_DIR)
print("CORPUS_PATH:", CORPUS_PATH)
print("OUT_DIR:", OUT_DIR)
print("HNSW_AVAILABLE:", HNSW_AVAILABLE)


CONFIG_PATH: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\src\configs\experiment_config.json
BASE_DIR: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código
CORPUS_PATH: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\corpus_rag_v1_index.jsonl
OUT_DIR: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\indexes\text2trade_nandina8_v1
HNSW_AVAILABLE: True


In [7]:
# =========================
# UTILIDADES (I/O, hashing, validación, snapshot del entorno)
# =========================

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


def write_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def is_8_digits(code: Any) -> bool:
    if code is None:
        return False
    s = str(code).strip()
    return bool(re.fullmatch(r"\d{8}", s))


def pick_text_field(row: Dict[str, Any], candidates: List[str]) -> Tuple[str, str]:
    # Retorna (campo_usado, texto). Prioriza campos curados para indexación.
    for k in candidates:
        v = row.get(k)
        if isinstance(v, str) and v.strip():
            return k, v
    # Fallback: concatenar strings largos como último recurso
    parts = []
    for k, v in row.items():
        if isinstance(v, str) and len(v) > 20:
            parts.append(v)
    return "fallback", "\n".join(parts)


def build_document_text(row: Dict[str, Any]) -> Tuple[str, Dict[str, Any]]:
    # Construye texto indexable: titulo + texto principal.
    title = row.get(TITLE_FIELD) if isinstance(row.get(TITLE_FIELD), str) else ""
    used_field, main_text = pick_text_field(row, TEXT_FIELD_CANDIDATES)
    text = (title.strip() + "\n" + main_text.strip()).strip() if title else main_text.strip()
    aux = {
        "text_field_used": used_field,
        "has_title": bool(title.strip())
    }
    return text, aux


def env_snapshot() -> Dict[str, Any]:
    snap = {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "machine": platform.machine(),
        "processor": platform.processor(),
        "numpy_version": np.__version__,
        "torch_version": getattr(torch, "__version__", None),
        "sentence_transformers_version": getattr(__import__("sentence_transformers"), "__version__", None),
        "hnswlib_available": HNSW_AVAILABLE,
    }
    return snap


In [8]:
# =========================
# CARGA DEL CORPUS + FILTRADO NANDINA-8 + DOCSTORE
# =========================

t0 = time.time()

corpus_sha256 = sha256_file(CORPUS_PATH)
config_sha256 = sha256_file(CONFIG_PATH)
rows = read_jsonl(CORPUS_PATH)

# Filtrar NANDINA-8
filtered = []
invalid_codes = 0
for r in rows:
    if r.get(TYPE_FIELD) != TARGET_TYPE:
        continue
    code = r.get(CODE_FIELD)
    if not is_8_digits(code):
        invalid_codes += 1
        continue
    filtered.append(r)

# Deduplicación defensiva por código (si existiera redundancia)
seen = set()
dedup = []
for r in filtered:
    code = str(r.get(CODE_FIELD)).strip()
    if code in seen:
        continue
    seen.add(code)
    dedup.append(r)

# Construir docstore
# doc_id estable: NANDINA:<codigo>
docstore_rows = []
texts = []
text_field_counter = {}

for i, r in enumerate(dedup):
    code = str(r.get(CODE_FIELD)).strip()
    doc_id = f"NANDINA:{code}"
    doc_text, aux = build_document_text(r)

    text_field_counter[aux["text_field_used"]] = text_field_counter.get(aux["text_field_used"], 0) + 1

    doc = {
        "doc_id": doc_id,
        "tipo": TARGET_TYPE,
        "codigo": code,
        "titulo": r.get(TITLE_FIELD, "") if isinstance(r.get(TITLE_FIELD, ""), str) else "",
        "texto_index": doc_text,
        "fuente": str(CORPUS_PATH.name),
        "offsets": {"source_row": i},
        "aux": aux,
    }
    docstore_rows.append(doc)
    texts.append(doc_text)

print(f"Total filas corpus: {len(rows)}")
print(f"Filtradas tipo=={TARGET_TYPE}: {len(filtered)}")
print(f"Códigos inválidos (no 8 dígitos): {invalid_codes}")
print(f"NANDINA-8 deduplicadas: {len(dedup)}")
print("Distribución de campo de texto usado:", text_field_counter)

DOCSTORE_PATH = STORE_DIR / "nandina8_docstore.jsonl"
write_jsonl(DOCSTORE_PATH, docstore_rows)

print("Docstore guardado en:", DOCSTORE_PATH)
print("Tiempo (s):", round(time.time() - t0, 2))


Total filas corpus: 7748
Filtradas tipo==nandina_8: 7644
Códigos inválidos (no 8 dígitos): 0
NANDINA-8 deduplicadas: 7644
Distribución de campo de texto usado: {'texto_index': 7644}
Docstore guardado en: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\indexes\text2trade_nandina8_v1\store\nandina8_docstore.jsonl
Tiempo (s): 0.38


In [9]:
# =========================
# EMBEDDINGS (bi-encoder) + PERSISTENCIA DEL MODELO
# =========================

# Cargar modelo
model = SentenceTransformer(BASE_BIENCODER)

# Guardar el modelo para reproducibilidad (congela la versión descargada/afinada)
model.save(str(MODEL_DIR))

# Embeddings deterministas para indexación (dropout desactivado)
# Nota: SentenceTransformer usa eval() internamente durante encode.
emb = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=EMBEDDING_NORMALIZE,
)

# Persistencia opcional de la matriz de embeddings
VECTORS_PATH = INDEX_DIR / "vectors.npy"
np.save(VECTORS_PATH, emb)

print("Embeddings shape:", emb.shape)
print("Embeddings guardados en:", VECTORS_PATH)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

C:\Users\VLADIMIR\miniconda3\envs\tesis-text2trade\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\VLADIMIR\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/239 [00:00<?, ?it/s]

Embeddings shape: (7644, 384)
Embeddings guardados en: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\indexes\text2trade_nandina8_v1\index\vectors.npy


In [10]:
# =========================
# CONSTRUCCIÓN DEL ÍNDICE ANN (HNSW) + MAPPING
# =========================

# Mapping posicional -> doc_id/código (para trazabilidad)
# Se asume que el orden de 'docstore_rows' coincide con el orden de 'emb'.
id_map = {}
for idx, d in enumerate(docstore_rows):
    id_map[str(idx)] = {
        "doc_id": d["doc_id"],
        "codigo": d["codigo"],
    }

ID_MAP_PATH = INDEX_DIR / "id_map.json"
with open(ID_MAP_PATH, "w", encoding="utf-8") as f:
    json.dump(id_map, f, ensure_ascii=False, indent=2)

print("ID map guardado en:", ID_MAP_PATH)

# Construir índice
INDEX_PATH = INDEX_DIR / "hnsw.index"

if HNSW_AVAILABLE:
    dim = emb.shape[1]
    num_elements = emb.shape[0]

    # hnswlib requiere float32
    emb_f32 = emb.astype(np.float32)

    p = hnswlib.Index(space=HNSW_SPACE, dim=dim)
    p.init_index(max_elements=num_elements, ef_construction=HNSW_EF_CONSTRUCTION, M=HNSW_M)
    p.add_items(emb_f32, np.arange(num_elements))
    p.set_ef(HNSW_EF_SEARCH)

    p.save_index(str(INDEX_PATH))
    print("Índice HNSW guardado en:", INDEX_PATH)
else:
    print("hnswlib no disponible. Se omitirá índice ANN y se usará búsqueda exacta en memoria.")


ID map guardado en: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\indexes\text2trade_nandina8_v1\index\id_map.json
Índice HNSW guardado en: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\indexes\text2trade_nandina8_v1\index\hnsw.index


In [11]:
# =========================
# CONFIGURACIÓN DE RECUPERACIÓN + METADATOS DE EJECUCIÓN
# =========================

retrieval_config = {
    "artifact_name": ARTIFACT_NAME,
    "target_type": TARGET_TYPE,
    "retriever": {
        "family": "dense_retrieval",
        "model": BASE_BIENCODER,
        "embedding_normalize": EMBEDDING_NORMALIZE,
        "batch_size": BATCH_SIZE,
    },
    "index": {
        "backend": "hnswlib" if HNSW_AVAILABLE else "exact",
        "space": HNSW_SPACE,
        "hnsw": {
            "M": HNSW_M,
            "ef_construction": HNSW_EF_CONSTRUCTION,
            "ef_search": HNSW_EF_SEARCH,
        },
    },
    "mcd": {
        "enabled": MCD_ENABLED,
        "passes": MCD_PASSES,
        "topk": MCD_TOPK,
        "aggregation": {
            "mean_sim_weight": MCD_MEAN_SIM_WEIGHT,
            "top3_freq_weight": MCD_TOP3_FREQ_WEIGHT,
        },
        "base_seed": MCD_BASE_SEED,
    },
}

with open(RETRIEVAL_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(retrieval_config, f, ensure_ascii=False, indent=2)

run_metadata = {
    "notebook": "05_Text2Trade_Indexacion_NANDINA.ipynb",
    "timestamp_unix": int(time.time()),
    "input": {
        "config_path": str(CONFIG_PATH),
        "config_sha256": config_sha256,
        "corpus_path": str(CORPUS_PATH),
        "corpus_sha256": corpus_sha256,
        "corpus_file": CORPUS_FILE,
        "filter": {
            "target_type": TARGET_TYPE,
            "type_field": TYPE_FIELD,
            "code_field": CODE_FIELD,
            "require_8_digits": True,
        },
        "text_fields_candidates": TEXT_FIELD_CANDIDATES,
    },
    "counts": {
        "corpus_rows": len(rows),
        "filtered_rows": len(filtered),
        "invalid_codes": invalid_codes,
        "indexed_docs": len(docstore_rows),
    },
    "model": {
        "base_biencoder": BASE_BIENCODER,
        "saved_model_dir": str(MODEL_DIR),
        "embedding_dim": int(emb.shape[1]),
    },
    "artifacts": {
        "out_dir": str(OUT_DIR),
        "docstore": str(DOCSTORE_PATH),
        "vectors": str(VECTORS_PATH),
        "index": str(INDEX_PATH) if HNSW_AVAILABLE else None,
        "id_map": str(ID_MAP_PATH),
        "retrieval_config": str(RETRIEVAL_CONFIG_PATH),
    },
    "environment": env_snapshot(),
    "policy": EXP_CONFIG.get("policy", {}),
    "version": EXP_CONFIG.get("version", None),
}

with open(RUN_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, ensure_ascii=False, indent=2)

print("retrieval_config guardado en:", RETRIEVAL_CONFIG_PATH)
print("run_metadata guardado en:", RUN_METADATA_PATH)


retrieval_config guardado en: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\indexes\text2trade_nandina8_v1\retrieval_config.json
run_metadata guardado en: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\indexes\text2trade_nandina8_v1\text2trade_nandina8_run_metadata.json


In [12]:
# =========================
# FUNCIONES DE CONSULTA (Top-k) + SANITY CHECK
# =========================

def cosine_topk_exact(query_vec: np.ndarray, matrix: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray]:
    # matrix y query_vec se asumen normalizados para coseno
    sims = matrix @ query_vec
    if k >= len(sims):
        idx = np.argsort(-sims)
    else:
        idx_part = np.argpartition(-sims, kth=k-1)[:k]
        idx = idx_part[np.argsort(-sims[idx_part])]
    return idx, sims[idx]


def retrieve_topk(query: str, k: int = 10) -> List[Dict[str, Any]]:
    q = model.encode([query], convert_to_numpy=True, normalize_embeddings=EMBEDDING_NORMALIZE)[0]
    q = q.astype(np.float32)

    if HNSW_AVAILABLE and INDEX_PATH.exists():
        p = hnswlib.Index(space=HNSW_SPACE, dim=int(emb.shape[1]))
        p.load_index(str(INDEX_PATH))
        p.set_ef(HNSW_EF_SEARCH)
        labels, distances = p.knn_query(q, k=k)
        # En espacio coseno, hnswlib retorna 'distances' ~ 1 - cosine_similarity
        labels = labels[0]
        distances = distances[0]
        sims = 1.0 - distances
        out = []
        for lab, sim in zip(labels, sims):
            d = docstore_rows[int(lab)]
            out.append({"codigo": d["codigo"], "doc_id": d["doc_id"], "score": float(sim)})
        return out

    # Fallback exacto
    idx, sims = cosine_topk_exact(q, emb.astype(np.float32), k=k)
    out = []
    for lab, sim in zip(idx, sims):
        d = docstore_rows[int(lab)]
        out.append({"codigo": d["codigo"], "doc_id": d["doc_id"], "score": float(sim)})
    return out


# --- MCD: estabilidad / incertidumbre ---

def embed_query_mcd(query: str, passes: int, base_seed: int) -> np.ndarray:
    # Activa dropout: train() para muestrear. No calcula gradientes.
    # Se fuerza un seed distinto por pasada para reproducibilidad.
    vecs = []
    model._first_module().train()  # activa dropout en el transformer subyacente
    for i in range(passes):
        torch.manual_seed(base_seed + i)
        np.random.seed(base_seed + i)
        with torch.no_grad():
            v = model.encode([query], convert_to_numpy=True, normalize_embeddings=EMBEDDING_NORMALIZE)[0]
        vecs.append(v)
    model._first_module().eval()  # restaura modo evaluación
    return np.stack(vecs, axis=0)


def retrieve_topk_mcd(query: str, k: int = 50, passes: int = 50) -> Dict[str, Any]:
    # Calcula estabilidad: promedio de similitud + frecuencia de aparición en Top-3
    q_vecs = embed_query_mcd(query, passes=passes, base_seed=MCD_BASE_SEED).astype(np.float32)

    top3_counter = {}
    mean_sims = None

    for i in range(passes):
        q = q_vecs[i]
        idx, sims = cosine_topk_exact(q, emb.astype(np.float32), k=k)
        if mean_sims is None:
            mean_sims = np.zeros(len(docstore_rows), dtype=np.float64)
        mean_sims[idx] += sims

        top3 = idx[:3]
        for j in top3:
            top3_counter[int(j)] = top3_counter.get(int(j), 0) + 1

    # Promedio de similitud (solo para docs que aparecieron en algún top-k: aproximación eficiente)
    # Para reranking final usamos candidatos union de top-k de la última pasada y top3_counter.
    # En un pipeline de evaluación se recomienda computar sobre un pool estable (p.ej., unión de top-k de todas las pasadas).

    # Construir pool de candidatos: top-k de la última pasada + top3 vistos
    pool = set(list(idx)) | set(top3_counter.keys())

    scored = []
    for doc_idx in pool:
        mean_sim = float(mean_sims[doc_idx] / passes) if mean_sims is not None else 0.0
        top3_freq = float(top3_counter.get(doc_idx, 0) / passes)
        stability = (MCD_MEAN_SIM_WEIGHT * mean_sim) + (MCD_TOP3_FREQ_WEIGHT * top3_freq)
        d = docstore_rows[int(doc_idx)]
        scored.append({
            "codigo": d["codigo"],
            "doc_id": d["doc_id"],
            "mean_sim": mean_sim,
            "top3_freq": top3_freq,
            "stability_score": stability,
        })

    scored.sort(key=lambda x: x["stability_score"], reverse=True)
    return {
        "query": query,
        "passes": passes,
        "topk": k,
        "results": scored[:k],
    }


# =========================
# PRUEBA RÁPIDA (SANITY CHECK)
# =========================
TEST_QUERY = "Computadora portátil con procesador Intel Core i5, memoria RAM 8 GB, disco sólido SSD 512 GB, pantalla LED de 14 pulgadas."

print("\n--- Top-10 (determinista) ---")
res = retrieve_topk(TEST_QUERY, k=10)
if pd is not None:
    display(pd.DataFrame(res))
else:
    for r in res:
        print(r)

if MCD_ENABLED:
    print("\n--- Top-10 (MCD / estabilidad) ---")
    res_mcd = retrieve_topk_mcd(TEST_QUERY, k=10, passes=MCD_PASSES)
    if pd is not None:
        display(pd.DataFrame(res_mcd["results"]))
    else:
        for r in res_mcd["results"]:
            print(r)

    # Guardar evidencia del sanity check (auditabilidad)
    sanity_out = EVAL_DIR / "smoke_test_results.json"
    with open(sanity_out, "w", encoding="utf-8") as f:
        json.dump({"deterministic": res, "mcd": res_mcd}, f, ensure_ascii=False, indent=2)
    print("\nSanity check guardado en:", sanity_out)



--- Top-10 (determinista) ---


,codigo,doc_id,score
0,85423100,NANDINA:85423100,0.565852
1,84717000,NANDINA:84717000,0.500783
2,85235100,NANDINA:85235100,0.444075
3,84663000,NANDINA:84663000,0.437581
4,85193010,NANDINA:85193010,0.414686
5,84732100,NANDINA:84732100,0.414331
6,84248230,NANDINA:84248230,0.398537
7,90303200,NANDINA:90303200,0.391226
8,84118200,NANDINA:84118200,0.379951
9,85415100,NANDINA:85415100,0.374321



--- Top-10 (MCD / estabilidad) ---


,codigo,doc_id,mean_sim,top3_freq,stability_score
0,85423100,NANDINA:85423100,0.565852,1.0,0.652682
1,84717000,NANDINA:84717000,0.500783,1.0,0.600626
2,85235100,NANDINA:85235100,0.444075,1.0,0.555260
3,84663000,NANDINA:84663000,0.437581,0.0,0.350065
4,85193010,NANDINA:85193010,0.414686,0.0,0.331749
5,84732100,NANDINA:84732100,0.414331,0.0,0.331465
6,84248230,NANDINA:84248230,0.398537,0.0,0.318829
7,90303200,NANDINA:90303200,0.391226,0.0,0.312981
8,84118200,NANDINA:84118200,0.379951,0.0,0.303961
9,85415100,NANDINA:85415100,0.374321,0.0,0.299457



Sanity check guardado en: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\indexes\text2trade_nandina8_v1\eval\smoke_test_results.json
